# Exercice 5 : Classification binaire avec KNN, SVM et Softmax
## TP1 — Vision par Ordinateur

---

**Objectif :** Classifier des données médicales (cancer du sein) avec trois algorithmes :
- **KNN** (K-Nearest Neighbors)
- **SVM** (Support Vector Machine)
- **Softmax** (Régression Logistique multinomiale)

---

## Étape 1 — Installation

In [ ]:
!pip install -q scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

print("Prêt !")

---
## Étape 2 — Charger le dataset

**Dataset Breast Cancer :**
- 569 échantillons (tumeurs)
- 30 caractéristiques numériques (taille, forme des cellules...)
- 2 classes : **malin** (0) / **bénin** (1)

In [ ]:
# Charger le dataset
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Nombre d'échantillons : {X.shape[0]}")
print(f"Nombre de features : {X.shape[1]}")
print(f"Classes : {data.target_names}")

## Étape 3 — Séparer train/test

In [ ]:
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train : {X_train.shape[0]} échantillons")
print(f"Test : {X_test.shape[0]} échantillons")

## Étape 4 — Normaliser les données

La normalisation met toutes les features à la même échelle (moyenne=0, écart-type=1).

**Pourquoi ?** Sans normalisation, les features avec de grandes valeurs domineraient les autres.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Données normalisées !")
print(f"Moyenne train : {X_train.mean(axis=0)[:3].round(2)}")
print(f"Écart-type train : {X_train.std(axis=0)[:3].round(2)}")

---
## Étape 5 — KNN (K-Nearest Neighbors)

**Principe :** Classer un point selon la majorité de ses K plus proches voisins.

**Exemple :** Si K=5 et que parmi les 5 voisins les plus proches il y a 3 bénins et 2 malins → on prédit bénin.

**Paramètre K :**
- K petit (ex: 1) → le modèle suit trop les données (surapprentissage)
- K grand (ex: 50) → le modèle est trop simple (sous-apprentissage)

In [ ]:
# Entraîner KNN avec K=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Prédire sur le jeu de test
y_pred_knn = knn.predict(X_test)

# Évaluer
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"Précision KNN (k=5) : {accuracy_knn:.2%}")

---
## Étape 6 — SVM (Support Vector Machine)

**Principe :** Trouver l'hyperplan (une ligne, un plan...) qui sépare au mieux les deux classes.

**Kernel :**
- `linear` → séparation linéaire (droite)
- `rbf` → séparation non-linéaire (courbe)

In [ ]:
# Entraîner SVM avec un noyau linéaire
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# Prédire
y_pred_svm = svm.predict(X_test)

# Évaluer
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Précision SVM (linéaire) : {accuracy_svm:.2%}")

---
## Étape 7 — Softmax (Régression Logistique)

**Principe :** Le modèle calcule un score (probabilité) pour chaque classe et choisit la plus élevée.

**Comment ça marche ?**
1. On calcule une combinaison linéaire des features
2. La fonction Softmax transforme ces scores en probabilités (qui somment à 1)
3. On choisit la classe avec la plus haute probabilité

**Softmax vs LogisticRegression :**
- Pour 2 classes, LogisticRegression avec `multi_class='multinomial'` utilise Softmax en interne
- C'est un classifieur linéaire simple et efficace

In [ ]:
# Entraîner Softmax (Régression Logistique multinomiale)
softmax = LogisticRegression(multi_class='multinomial', max_iter=1000)
softmax.fit(X_train, y_train)

# Prédire
y_pred_softmax = softmax.predict(X_test)

# Évaluer
accuracy_softmax = accuracy_score(y_test, y_pred_softmax)
print(f"Précision Softmax : {accuracy_softmax:.2%}")

---
## Étape 8 — Matrices de confusion

La matrice de confusion montre les prédictions correctes et les erreurs :
- **Diagonale** = bonnes prédictions
```
                    Prédit: Malin  Prédit: Bénin
Vrai: Malin         [VP]            [FN]
Vrai: Bénin         [FP]            [VN]
```

In [ ]:
# Matrice de confusion pour chaque classificateur
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# KNN
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn, ax=axes[0])
axes[0].set_title(f'KNN\nPrécision : {accuracy_knn:.2%}', fontsize=14)

# SVM
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[1])
axes[1].set_title(f'SVM\nPrécision : {accuracy_svm:.2%}', fontsize=14)

# Softmax
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_softmax, ax=axes[2])
axes[2].set_title(f'Softmax\nPrécision : {accuracy_softmax:.2%}', fontsize=14)

plt.tight_layout()
plt.show()

---
## Étape 9 — Comparaison des 3 classifieurs

In [ ]:
# Comparaison sous forme de tableau
print("=" * 50)
print("COMPARAISON DES 3 CLASSIFIEURS")
print("=" * 50)
print(f"KNN (k=5)    : {accuracy_knn:.2%}")
print(f"SVM          : {accuracy_svm:.2%}")
print(f"Softmax      : {accuracy_softmax:.2%}")
print("=" * 50)

# Trouver le meilleur
scores = {'KNN': accuracy_knn, 'SVM': accuracy_svm, 'Softmax': accuracy_softmax}
meilleur = max(scores, key=scores.get)
print(f"\nLe meilleur classifieur est : {meilleur} ({scores[meilleur]:.2%})")

In [ ]:
# Graphique comparatif
noms = ['KNN', 'SVM', 'Softmax']
precisions = [accuracy_knn, accuracy_svm, accuracy_softmax]
couleurs = ['#3498db', '#2ecc71', '#e74c3c']

plt.figure(figsize=(10, 6))
bars = plt.bar(noms, precisions, color=couleurs, width=0.5)
plt.ylim(0.9, 1.0)
plt.ylabel('Précision')
plt.title('Comparaison des 3 classifieurs', fontsize=16)

# Ajouter les valeurs au-dessus des barres
for bar, p in zip(bars, precisions):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{p:.2%}', ha='center', fontsize=13, fontweight='bold')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

---
## Résumé

| Algorithme | Principe | Avantage | Inconvénient |
|------------|----------|----------|--------------|
| **KNN** | K plus proches voisins | Simple, rapide | Lent en prédiction |
| **SVM** | Hyperplan optimal | Précis, efficace | Moins interprétable |
| **Softmax** | Probabilités par classe | Simple, probabiliste | Linéaire uniquement |

**Étapes clés d'une classification :**
1. Charger les données
2. Séparer train/test
3. Normaliser
4. Entraîner
5. Évaluer avec la précision et la matrice de confusion